# Semenov et al. (2026) from dandiset 001525: membrane recovery after 1-us electroporation

**Paper.** Semenov, Silkuniene, Silkunas, Pakhomova, Bixler, Ibey, Kochunov & Pakhomov (2026),
"Sub-microsecond optical measurements of cell membrane charging and lesioning by pulsed electric
fields", Bioelectrochemistry 171, 109283, doi:10.1016/j.bioelechem.2026.109283.

**Dataset.** [DANDI:001525](https://dandiarchive.org/dandiset/001525/0.261006.1856), version
`0.261006.1856` (doi:10.48324/dandi.001525/0.261006.1856), streamed from the archive. The paper
cites no version (its data statement says data are available on request); this is the most
recent and only published version, and its content (73 files, 2.51 GB) is the same as the draft's.

**Claims file.** `claims/001525/10.1016-j.bioelechem.2026.109283.yaml`, written from the version of
record (main text, Methods, legends; no supplement).

**What the dataset holds.** Ten CHO cells loaded with Di-8-ANEPPS, 73 sets of 100 strobe images:
per cell a single 50 V pulse and conditioning pulses of 100 to 350 V each followed by a 50 V test
pulse, after 1 us (cells c1 to c4) or 5 us (c5 to c10). This is the two-pulse protocol of
Section 3.7 and Fig. 7A and 7B (bottom). Nothing else of the paper is here.

| Claim | Tested here |
|---|---|
| fig7a-test-pulse-cosine-up-to-1000 | tested |
| fig7b-test-pulse-unchanged-to-760 | tested |
| fig7b-test-pulse-reduced-1200-1400 | tested |
| fig7c-full-reseal-mild-residual-strong | tested in part: its test-pulse parts (the pole difference during the pulse needs the missing Fig. 4B images) |
| fig5b-conductivity-calibration | tested (a computation from Eq. 2, no recording) |
| fig7b-conditioning-pulse-below-linear | blocked, data missing (Fig. 4B images) |
| fig7c-conductivity-drops-10-fold | blocked, depends on the claim above |
| fig1, fig2 (6 claims), fig3 (2), fig4 (6), fig5a (2), fig5c, fig6 (2) | blocked, data missing |

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
import strobe as S

assets = S.list_assets()
meta = S.read_all_meta(assets)
subjects = sorted({d["subject"] for d in meta}, key=lambda s: int(s.split("_c")[1]))
print(f"dandiset {S.DANDISET_ID} version {S.DANDISET_VERSION}: {len(assets)} assets, "
      f"{sum(a['size'] for a in assets)/1e9:.2f} GB, {len(subjects)} subjects")

In [ ]:
# Data check: what the files hold, against the paper's description
df = pd.DataFrame([{k: d[k] for k in ("subject", "kind", "cond_v", "test_v", "interval_us",
                     "shape", "rate", "unit", "electrode_gap", "dye")} for d in meta])
df["shape"] = df["shape"].astype(str)
print("files per subject and protocol (conditioning volts; 'single' = one 50 V pulse):")
tab = df.assign(level=df.cond_v.fillna(0).astype(int).astype(str).replace("0", "single"))
print(pd.crosstab([tab.subject, tab.interval_us.fillna(0).astype(int)], tab.level).to_string())
print("\nimage stack shapes:", Counter(df["shape"]))
print("frame rate (Hz):", Counter(df.rate), "| data unit:", Counter(df.unit))
print("test / single pulse volts:", Counter(df.test_v), "| interval (us):", Counter(df.interval_us.dropna()))
print("electrode gap stated:", Counter(df.electrode_gap), "(paper, Methods 2.2: spacing 1.6 mm)")
print("dye:", Counter(df.dye))
print("\nseries description:", meta[0]["series_description"])
print("\ngeneral description in every file (identical:",
      len({d['description'].split('####')[-1] for d in meta}) == 1, "):")
print(meta[0]["description"].split("#####")[-1][:700])
# Field strengths. The files give generator volts; the Fig. 7A panels give seven strengths, and the
# files hold seven levels. The rank mapping (STRENGTH_V_CM) against the alternatives the files suggest:
volts = np.array([50, 100, 150, 200, 250, 300, 350])
rank = np.array([S.STRENGTH_V_CM[L] for L in S.LEVELS])
print("\nrank mapping, V/cm per generator volt:", dict(zip(volts.tolist(), np.round(rank / volts, 2).tolist())))
desc_kv = [0.15, 0.52, 0.9, 1.27, 1.57]
print("strengths in the general description (kV/cm):", desc_kv, "- five values for seven levels")
slope = (desc_kv[1] - desc_kv[0]) / 100          # read as 50, 150, 250, 350 V in steps of 100 V
desc_scale = 1000 * (desc_kv[0] + slope * (volts - 50))
print("if the description's steps of 0.37 kV/cm are 100 V apart from 0.15 kV/cm at 50 V, the levels are",
      np.round(desc_scale).astype(int).tolist(), "V/cm; rank mapping", rank.tolist(),
      f"(ratio {np.round(desc_scale / rank, 2).min()} to {np.round(desc_scale / rank, 2).max()})")
print("a uniform field over the stated 1.00 mm gap would give", (volts / 0.1).astype(int).tolist(),
      "V/cm; over the paper's 1.6 mm,", np.round(volts / 0.16).astype(int).tolist())

In [ ]:
# Load and reduce every file: cell, trace, collapsed images, dF profiles under each reading.
results = S.process_subjects(meta, subjects, workers=10)
five = [s for s in subjects if all(o["interval_us"] in (None, 5) for o in results[s])]
one = [s for s in subjects if s not in five]
print("5-us cells (Fig. 7):", five)
print("1-us cells (not in the paper):", one)
# Recordings deposited twice: compare a digest of each image stack
dig = Counter(round(o["data_digest"]) for s in subjects for o in results[s])
print("files:", sum(len(results[s]) for s in subjects), "| repeated image stacks:",
      sum(v - 1 for v in dig.values() if v > 1))
rows = []
for s in subjects:
    for o in results[s]:
        rows.append(dict(subject=s.split("_")[-1], level=S.level_of(o), cx=o["cx"], cy=o["cy"],
                         r_px=o["r_membrane"], n_dark=o["n_dark"], dark=str(o["dark_idx"])))
cells = pd.DataFrame(rows)
print("\ncell position (px) and membrane radius, range over a subject's files:")
print(cells.groupby("subject", sort=False).agg(cx_min=("cx", "min"), cx_max=("cx", "max"),
      cy_min=("cy", "min"), cy_max=("cy", "max"), r_min=("r_px", "min"), r_max=("r_px", "max"),
      dark_frames=("n_dark", "sum")).round(1).to_string())
print("\ndark (no-flash) frames per file:", Counter(cells.n_dark), "| at frame index:",
      Counter(i for s in subjects for o in results[s] for i in o["dark_idx"]).most_common(8))
fm = np.array([o["frame_mean"] for s in subjects for o in results[s]])
dk = np.concatenate([S.dark_frames(o["frame_mean"][:, None, None]) for s in subjects for o in results[s]])
print(f"field mean of dark frames {fm.ravel()[dk].mean():.0f} counts, of lit frames "
      f"{fm.ravel()[~dk].mean():.0f}; pixel range "
      f"{min(o['frame_mean'].min() for s in subjects for o in results[s]):.0f} to "
      f"{max(o['frame_mean'].max() for s in subjects for o in results[s]):.0f} (field means)")

In [ ]:
# Recording order: each subject's files by session start time. A control recorded after the strong
# conditioning pulses would come from a cell that had already been electroporated.
lab = lambda o: "single" if o["kind"] == "single" else f"{o['cond_v']}V"
for s in subjects:
    fs = sorted(results[s], key=lambda o: o["start"])
    print(f"{s.split('_')[-1]:>4}: " + " ".join(lab(o) for o in fs))
pos = {s.split("_")[-1]: [i for i, o in enumerate(sorted(results[s], key=lambda o: o["start"]))
                          if o["kind"] == "single"] for s in subjects}
print("position of the single-pulse file(s) in the recording order (0 = first):", pos)
print("conditioning levels recorded in rising order in every subject:",
      all([o["cond_v"] for o in sorted(results[s], key=lambda o: o["start"]) if o["kind"] == "pair"]
          == sorted(o["cond_v"] for o in results[s] if o["kind"] == "pair") for s in subjects))

In [ ]:
# Field direction and clock. The files do not give the field direction in image coordinates;
# every file's profile is fit with a cosine and the phases are pooled (5-us cells).
MAIN = ("dF", True, True)          # dark frames dropped, annulus background (see below)
READINGS = {("dF", d, b): f"dark {'dropped' if d else 'kept'}, background {'annulus' if b else 'none'}"
            for d in (False, True) for b in (False, True)}
READINGS[("dF_peak", True, True)] = "dark dropped, background annulus, first-round peak trace"
th = np.arange(360) + 0.5
res5 = {s: results[s] for s in five}
PHI = {}
for key, name in READINGS.items():
    PHI[key], phis = S.field_angle(res5, key)
    print(f"{name}: depolarized pole at image angle {PHI[key]:.1f} deg, per-file phases "
          f"{np.min(phis):.0f} to {np.max(phis):.0f} (sd {np.std(phis):.1f})")
phi = PHI[MAIN]
# Clock: the first 40 images are baseline and the next 60 carry the pulse (Methods 2.3). Per file,
# the pole contrast (cathodic minus anodic trace intensity over their mean) is split at every
# frame k; the split with the largest t statistic should be 40.
best, tmax = [], []
for s in subjects:
    for o in results[s]:
        ft = np.array([S.rotate_to_field(f, phi) for f in o["frame_trace"]])
        c, a = np.array([S.pole_means(f) for f in ft]).T
        ok = ~S.dark_frames(o["frame_mean"][:, None, None])
        con = (c - a) / ((c + a) / 2)
        ts = []
        for k in range(10, 91):
            x0, x1 = con[:k][ok[:k]], con[k:][ok[k:]]
            ts.append(stats.ttest_ind(x1, x0).statistic)
        best.append(10 + int(np.argmax(ts))); tmax.append(np.max(ts))
best = np.array(best)
print(f"\nbest split frame over {len(best)} files: at 40 in {np.sum(best == 40)}, within 39 to 41 in "
      f"{np.sum(abs(best - 40) <= 1)}; others {sorted(best[abs(best - 40) > 1].tolist())}")
print(f"t statistic at the best split: median {np.median(tmax):.1f}, min {np.min(tmax):.1f}")
print("The series description says exposures began 1 us after the start of recording; the images "
      "show the pulse from frame 40 (6.7 s at 6 Hz).")
# Field direction from the control (single-pulse) files only, which carry no conditioning damage,
# for the variant that does not take the direction from the profiles it then tests.
PHI_CTRL, phis_c = S.field_angle({s: [o for o in results[s] if o["kind"] == "single"] for s in five}, MAIN)
print(f"main reading, control files only: depolarized pole at {PHI_CTRL:.1f} deg (per-file sd "
      f"{np.std(phis_c):.1f}; all files {PHI[MAIN]:.1f})")

In [ ]:
# The analysed cell and its trace in each subject's first file (collapsed baseline image). Orange:
# the trace used (intensity peak within 8 px inside the cell's outer edge); blue: the first round's
# trace (plain intensity peak along each ray), kept as a variant. The distance of each trace inside
# the outer edge (steepest outward fall of intensity) is printed for every file.
fig, ax = plt.subplots(2, 5, figsize=(16, 6.6))
ang = np.deg2rad(np.arange(S.N_POINTS) + 0.5)
for a, s in zip(ax.ravel(), subjects):
    o = sorted(results[s], key=lambda o: o["start"])[0]
    x0, y0 = o["crop_origin"]
    a.imshow(o["crop"], cmap="gray")
    for r_, col in ((o["rpk_peak"], "C0"), (o["rpk"], "C1")):
        a.plot(o["cx"] - x0 + np.cos(ang) * r_, o["cy"] - y0 + np.sin(ang) * r_, col, lw=0.7)
    a.set_title(f"{s.split('_')[-1]} ({'5' if s in five else '1'} us), r = {o['r_membrane']:.1f} px", fontsize=9)
    a.axis("off")
plt.show()
rows = []
for s in subjects:
    for o in results[s]:
        x0, y0 = o["crop_origin"]
        if o is sorted(results[s], key=lambda o: o["start"])[0]:
            e = S.outer_edge(o["crop"], o["cx"] - x0, o["cy"] - y0, o["r0"])
            rows.append(dict(cell=s.split("_")[-1], inside_edge_used=np.median(e - o["rpk"]),
                             inside_edge_peak=np.median(e - o["rpk_peak"]),
                             frac_peak_over_4px_inside_used=np.mean(o["rpk"] - o["rpk_peak"] > 4)))
print("first file of each subject: median distance of each trace inside the outer edge (px), and the fraction "
      "of angles where the first round's trace lies more than 4 px inside the trace used")
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("image crops: intensity", f"{min(o['crop'].min() for s in subjects for o in results[s]):.0f} to "
      f"{max(o['crop'].max() for s in subjects for o in results[s]):.0f} counts (grey scale per panel)")

In [ ]:
# Where the trace lies at the poles, and whether its notches reach the pole arcs (5-us cells, every
# file). Depth: how far the trace lies inside the cell's outer edge, median over the +-5 degree pole
# arc, for the trace used (8 px window) and the window variants. A depth near the window's width
# means the peak sits at the window's inner limit, so the window, not the image, places the trace.
# Notch: angles where the trace used falls more than 2 px below its running median over 61 degrees.
fa = (np.arange(S.N_POINTS) + 0.5 - PHI[MAIN]) % 360           # angle to the field of each image angle
dpole = np.minimum(np.abs((fa + 180) % 360 - 180), np.abs(fa - 180))
arc_c, arc_a = np.abs((fa + 180) % 360 - 180) <= S.POLE_HALF_WIDTH_DEG, np.abs(fa - 180) <= S.POLE_HALF_WIDTH_DEG
from scipy import ndimage as _nd
rows, notch_rows = [], []
for s in five:
    row = dict(cell=s.split("_")[-1])
    for w in (4, 6, 8, 10, 12):
        dep = [o["edge"] - (o["rpk"] if w == S.EDGE_WINDOW_PX else o["rpk_w"][w]) for o in results[s]]
        row[f"cath_w{w}"] = np.median([d[arc_c].mean() for d in dep])
        row[f"anod_w{w}"] = np.median([d[arc_a].mean() for d in dep])
    notch_d = []
    for o in results[s]:
        dev = o["rpk"] - _nd.median_filter(o["rpk"], 61, mode="wrap")
        notch_d.append(dpole[dev < -2].min() if (dev < -2).any() else np.nan)
        for pole, arc in (("cathodic", arc_c), ("anodic", arc_a)):
            if (dev[arc] < -2).any():
                notch_rows.append(dict(cell=s.split("_")[-1], strength=S.STRENGTH_V_CM[S.level_of(o)], pole=pole,
                                       trace_below_running_median_px=dev[arc].min(),
                                       depth_inside_edge_px=(o["edge"] - o["rpk"])[arc].mean(),
                                       pole_dF=S.pole_means(S.rotate_to_field(o[MAIN], PHI[MAIN]))[0 if pole == "cathodic" else 1]))
    row["files_with_notch"] = int(np.sum(~np.isnan(notch_d)))
    row["notch_nearest_pole_deg"] = np.nanmin(notch_d) if np.any(~np.isnan(notch_d)) else np.nan
    rows.append(row)
print("median depth of the trace inside the outer edge in the pole arcs (px), per window width, and notches:")
print(pd.DataFrame(rows).round(1).to_string(index=False))
o = sorted(results[five[0]], key=lambda o: o["start"])[0]
dev = o["rpk"] - _nd.median_filter(o["rpk"], 61, mode="wrap")
print(f"{five[0].split('_')[-1]}, first file: notch at angles to the field {sorted(np.round(fa[dev < -2]).astype(int).tolist())} "
      f"(cathodic pole 0, anodic 180; the pole arcs are +-{S.POLE_HALF_WIDTH_DEG} degrees)")
print("files whose notch reaches a pole arc (main reading pole dF, %):")
print(pd.DataFrame(notch_rows).round(1).to_string(index=False) if notch_rows else "none")


In [ ]:
# Polarization scale under each reading, against the paper's: Fig. 7A shows the 150/150 profile
# at about +-13% dF, and Fig. 2C at about +-11% at 150 V/cm; Results 3.3 give Di-8-ANEPPS 9.1% dF
# per 100 mV, and 1.5 E R at 150 V/cm and R = 8 um is 180 mV, about 16% dF.
for key, name in READINGS.items():
    P = S.profiles_by_level(results, five, key, PHI[key])
    A, ph, r2, _ = S.cosine_fit(th, P["single"].mean(0))
    off = P["single"].mean()
    print(f"{name}: 150/150 cosine amplitude {A:.1f}% dF, profile mean {off:.2f}% dF")
print("Background (annulus) in counts, baseline image, 5-us files: ",
      np.round(np.percentile([o[('bg', True, True)][0] for s in five for o in results[s]], [0, 50, 100])))
print("Trace intensity (baseline, mean over perimeter), counts: ",
      np.round(np.percentile([o['frame_trace'][1:40].mean() for s in five for o in results[s]], [0, 50, 100])))

In [ ]:
# Each cell's own cosine fit (main reading), and where the uniform offset in dF comes from.
prof_main = S.profiles_by_level(results, five, MAIN, PHI[MAIN])
rows = []
for i, s in enumerate(five):
    for L in S.LEVELS:
        A, ph, r2, _ = S.cosine_fit(th, prof_main[L][i])
        rows.append(dict(cell=s.split("_")[-1], strength=S.STRENGTH_V_CM[L], A=round(A, 1),
                         phase=round(ph, 1), mean=round(prof_main[L][i].mean(), 2)))
pc = pd.DataFrame(rows)
for col, what in (("A", "cosine amplitude, % dF"), ("phase", "phase of the cell's own fit, degrees from the pooled direction"),
                  ("mean", "profile mean, % dF")):
    print(pc.pivot(index="cell", columns="strength", values=col).loc[[s.split('_')[-1] for s in five]].to_string(),
          f"\n({what})\n")
# Baseline trend: trace intensity minus annulus background per lit frame, a line over frames 0 to 39,
# slope in % of the baseline mean per frame. Extrapolated to the lit pulse frames (mean frame index
# about 69.5 against 19.5), it gives the uniform dF that drift alone predicts.
rows = []
for s in five:
    for o in results[s]:
        ok = ~S.dark_frames(o["frame_mean"][:, None, None])
        y = o["frame_trace"].mean(1) - o["frame_bg"]
        k, kp = np.flatnonzero(ok[:40]), 40 + np.flatnonzero(ok[40:])
        sl = np.polyfit(k, y[k], 1)[0]
        rows.append(dict(cell=s.split("_")[-1], strength=S.STRENGTH_V_CM[S.level_of(o)],
                         slope=100 * sl / y[k].mean(), drift_pred=100 * sl * (kp.mean() - k.mean()) / y[k].mean(),
                         mean_dF=o[MAIN].mean()))
tr = pd.DataFrame(rows)
print("baseline slope (% per frame), the uniform dF it predicts at the pulse frames, and the measured profile mean:")
print(tr.groupby("strength")[["slope", "drift_pred", "mean_dF"]].mean().round(2).to_string())
print(f"\n{len(tr)} files: slope {tr.slope.mean():.3f}% per frame (range {tr.slope.min():.3f} to {tr.slope.max():.3f}), "
      f"slope < 0 in {int((tr.slope < 0).sum())}; predicted uniform dF {tr.drift_pred.mean():.2f}%, measured "
      f"profile mean {tr.mean_dF.mean():.2f}%; correlation over files r = {np.corrcoef(tr.drift_pred, tr.mean_dF)[0, 1]:.2f}")

**Readings used throughout.** Details the paper does not fix could move the numbers, and every
claim below is shown under each:

- the trace: the paper draws "the 6-pixel-wide trace ... along the cell perimeter" (Methods 2.3).
  Here (main) it is the intensity peak within 8 pixels inside the cell's outer edge; the first
  round's trace, the plain intensity peak along each ray, is a variant. The overlays above show
  that the plain peak falls on bright dye inside cells c5 and c9 (a median 11.7 and 8.9 px inside
  the outer edge, against 2.5 to 4.8 px for the trace used in every cell). The 8-pixel window is a
  choice; windows of 4 to 12 pixels are run as variants (section "Sensitivity to the trace window").
  In c5 the window places the cathodic trace (median depth 3.2 to 10.7 px inside the edge for
  windows of 4 to 12 px); it moves the cathodic trace of c9 by 3.6 px, and in the other cells the
trace by at most 2.2 px at a pole. The
  trace used has notches; one reaches a pole arc in three files: c5 at 760 and 1000 V/cm at the
  cathodic pole (8.8 and 8.2 px below its running median; pole dF -3.3% and -1.9%) and c9 at
  360 V/cm at the anodic pole (2.7 px). In c5's first file, the one in the overlay, the notch lies
  at 325 to 338 degrees, 22 to 35 degrees from the cathodic pole, outside its arc.
- background: (a) none, as Methods 2.3 describe for the potentiometric images; (b) the median of
  an annulus 1.6 to 2.2 cell radii from the centre, a cell-free region as Methods 2.4 use for Fluo-4.
- dark frames: (a) kept, (b) dropped. Images with no laser flash are in most files (frame 0).
- the cell's control when a cell has two single-pulse files: their mean, the first, the last.
- the cells: all six 5-us cells (the paper has n = 5 and does not say which), and each set of five.
- the field direction: pooled over all 5-us files (main), or from the control files only.

The main reading, adopted as an assumption, is the edge trace, background (b) and dark frames
dropped: the paper's own scale (Fig. 7A about 13% dF at 150/150, and 9.1% dF per 100 mV with a
150 V/cm IMP near 180 mV) is met only with a background removed (output above), and an image with
no flash holds no fluorescence.

**The control.** The Fig. 7A legend and the claims file name a 150/150 pulse pair as the control.
The files hold no 50 V + 50 V pair; the control here is each cell's single 50 V pulse, recorded
first in every cell (output above), so it comes from an undamaged cell. An assumption.

**Field strengths.** The conditioning strengths are mapped to the dataset's volts by rank
(`STRENGTH_V_CM`), an assumption: the Fig. 7A panels name seven strengths and the files hold
seven levels. The general description's five strengths cannot be matched to seven levels; read
as steps of 100 V from 0.15 kV/cm at 50 V they would put the levels 0 to 11% below the rank
mapping (350 V at 1.26 kV/cm), which changes no level's place in the order the claims test. Neither
the stated 1.00 mm gap nor the paper's 1.6 mm gives the paper's 150 V/cm at 50 V as a uniform field.

**The uniform offset.** The profiles carry a negative mean dF (-4.3% at the control, main
reading). The baseline images fade: the trace intensity falls by about 0.10% per frame over
frames 0 to 39 in 43 of 44 files, which over the 50 frames between the baseline and pulse
centres predicts a uniform dF of about -5%. The sign is that of the dye bleaching the paper
reports for Di-8-ANEPPS (Results 3.2: bleaching took about 5% from dF at the cathodic pole and
added about 5% at the anodic). The size does not match: the fade predicts -5.16% over all files
against a measured profile mean of -3.67%, -6.96% against -3.58% at the control and -4.68%
against -1.81% at 760 V/cm, and r = 0.18 over files. The source of the offset is not established.

In [ ]:
def poles_for(key, subs=None, benchmark="mean", phi=None):
    subs = subs or five
    return S.pole_values(S.profiles_by_level(results, subs, key, PHI[key] if phi is None else phi, benchmark))

def pct_of_control(poles, L, pole):
    return 100 * S.polarization(poles, L, pole).mean() / S.polarization(poles, "single", pole).mean()

def test_vs_control(poles, L, pole, kind):
    x, c = S.polarization(poles, L, pole), S.polarization(poles, "single", pole)
    if kind == "paired":
        return stats.ttest_rel(x, c).pvalue
    if kind == "unpaired":
        return stats.ttest_ind(x, c).pvalue
    return stats.ttest_1samp(100 * x / c, 100).pvalue   # variant: per-cell normalized, one-sample

POLES = ("cathodic", "anodic", "mean")
# The paper's test (claims file, fig7b-test-pulse-unchanged-to-760, as amended after the reader's
# answer in questions/001525.yaml): the unpaired two-sample Student's two-sided t-test of per-cell
# dF against the control group, for the Fig. 7B legend's comparison "from the undamaged membrane
# (gray area between horizontal dashed lines)". The paired and one-sample tests are variants.
TEST = "unpaired"
TESTS = ("unpaired", "paired")      # the paper's test first, then the paired variant
# Variants: (reading, control when a cell has two single-pulse files, cells, field direction from).
VARIANTS = [(key, b, None, "all") for key in READINGS for b in ("mean", "first", "last")] + \
           [(MAIN, "mean", tuple(s for s in five if s != drop), "all") for drop in five] + \
           [(MAIN, "mean", None, "control")]
def vname(v):
    key, b, subs, ph = v
    cells_ = "6 cells" if subs is None else "without " + [s for s in five if s not in subs][0].split("_")[-1]
    return f"{READINGS[key]}; control {b}; {cells_}" + ("; field from controls" if ph == "control" else "")
def vargs(v):
    key, b, subs, ph = v
    return dict(key=key, subs=list(subs) if subs else None, benchmark=b, phi=PHI_CTRL if ph == "control" else None)
P0 = poles_for(MAIN)
print(f"{len(VARIANTS)} variants. Main reading: per-cell pole dF (%) by conditioning strength (V/cm)")
for L in S.LEVELS:
    c, a = P0[L]
    print(f"{S.STRENGTH_V_CM[L]:>5}: cathodic {np.round(c, 1)}  anodic {np.round(a, 1)}")

## fig7a-test-pulse-cosine-up-to-1000

> After a conditioning pulse of up to 1000 V/cm, polarization by the 150 V/cm test pulse 5 us
> later still follows a cosine of the angle to the field, while 1200 and 1400 V/cm conditioning
> pulses markedly reduce charging at the cell poles.

Per strength the mean profile over cells is fit with A cos(theta - phi). Reading (a): at each pole
the fit lies within the s.e. of the mean pole dF. Reading (b): pole polarization does not differ
from the benchmark by the two-sided t-test of Fig. 7B, which the claims file (amended after the
reader's answer, questions/001525.yaml) takes as the unpaired two-sample t-test against the
control group; the paired test is shown as a variant. Hindered at 1200 and 1400 V/cm: pole
polarization below the benchmark at p < 0.05 by that test. Pole readings as in Fig. 7B: cathodic,
anodic, mean of both.

In [ ]:
def fig7a_eval(key, subs=None, benchmark="mean", phi=None, verbose=False):
    subs = subs or five
    prof = S.profiles_by_level(results, subs, key, PHI[key] if phi is None else phi, benchmark)
    poles = S.pole_values(prof)
    fitinfo, band_ok = {}, {}
    for L in S.LEVELS:
        mp = prof[L].mean(0)
        A, ph, r2, fit = S.cosine_fit(th, mp)
        fc, fa = S.pole_means(fit)
        c, a = poles[L]
        sec, sea = c.std(ddof=1) / np.sqrt(len(c)), a.std(ddof=1) / np.sqrt(len(a))
        okc, oka = abs(c.mean() - fc) <= sec, abs(a.mean() - fa) <= sea
        # diagnostic only, not the criterion: the same band with an offset term, A cos(theta - phi) + c0
        t = np.deg2rad(th); X = np.c_[np.cos(t), np.sin(t), np.ones_like(t)]
        fo = X @ np.linalg.lstsq(X, mp, rcond=None)[0]
        foc, foa = S.pole_means(fo)
        fitinfo[L] = (A, ph, r2, c.mean() - fc, sec, a.mean() - fa, sea, c.mean() - foc, a.mean() - foa,
                      abs(c.mean() - foc) <= sec and abs(a.mean() - foa) <= sea)
        band_ok[L] = {"cathodic": okc, "anodic": oka, "mean": okc and oka}
    low = [L for L in S.LEVELS if S.STRENGTH_V_CM[L] <= 1000]
    high = [300, 350]
    out = {}
    for pole in POLES:
        for test in TESTS:
            hind = all(test_vs_control(poles, L, pole, test) < 0.05 and
                       S.polarization(poles, L, pole).mean() < S.polarization(poles, "single", pole).mean()
                       for L in high)
            cos_a = all(band_ok[L][pole] for L in low)
            cos_b = all(test_vs_control(poles, L, pole, test) >= 0.05 for L in low[1:])
            out[(pole, test)] = (cos_a, cos_b, hind)
    if verbose:
        print("strength  A(%dF)  phase  R2    cath: mean-fit  s.e. | anod: mean-fit  s.e. | with offset term: cath, anod, within s.e.")
        for L in S.LEVELS:
            A, ph, r2, dc, sec, da, sea, doc, doa, ok_o = fitinfo[L]
            print(f"{S.STRENGTH_V_CM[L]:>6} {A:6.2f} {ph:6.1f} {r2:5.2f}   {dc:8.2f} {sec:5.2f}   | {da:8.2f} {sea:5.2f}"
                  f"   | {doc:6.2f} {doa:6.2f} {ok_o}")
    return out, fitinfo

out, fitinfo = fig7a_eval(MAIN, verbose=True)
print("\nMain reading: (pole, test): follows cosine by (a) s.e. band, (b) no difference; hindered at 1200 and 1400")
for k, v in out.items():
    tag = "the paper's test" if k[1] == TEST else "variant"
    print(k, tag, v, "-> holds under (a)" if v[0] and v[2] else "", "-> holds under (b)" if v[1] and v[2] else "")
P = P0
print("\np values against the benchmark, main reading (unpaired, paired):")
for pole in POLES:
    print(pole, {S.STRENGTH_V_CM[L]: (round(float(test_vs_control(P, L, pole, 'unpaired')), 4),
                 round(float(test_vs_control(P, L, pole, 'paired')), 4)) for L in S.LEVELS[1:]})

In [ ]:
fig, ax = plt.subplots(1, 7, figsize=(17, 2.8), sharey=True)
prof = S.profiles_by_level(results, five, MAIN, PHI[MAIN])
bench = S.cosine_fit(th, prof["single"].mean(0))[3]
lo_, hi_ = np.inf, -np.inf
for i, L in enumerate(S.LEVELS):
    m, se = prof[L].mean(0), prof[L].std(0, ddof=1) / np.sqrt(len(five))
    ax[i].fill_between(th, m - se, m + se, color="0.8"); ax[i].plot(th, m, "k", lw=0.6)
    ax[i].plot(th, S.cosine_fit(th, m)[3], "C1"); ax[i].plot(th, bench, "C1--", lw=0.8)
    ax[i].set_title(f"{S.STRENGTH_V_CM[L]}/150"); ax[i].set_xticks([0, 180, 360])
    lo_, hi_ = min(lo_, (m - se).min()), max(hi_, (m + se).max())
ax[0].set_ylabel("dF, % (main reading)"); ax[3].set_xlabel("angle to the field, degrees")
plt.show()
print(f"plotted mean +- s.e. ranges from {lo_:.1f} to {hi_:.1f}% dF; all points within the axes")

In [ ]:
# Every reading
rows = []
for v in VARIANTS:
    o, fi = fig7a_eval(**vargs(v))
    ou = {k: x for k, x in o.items() if k[1] == TEST}        # the paper's test
    op = {k: x for k, x in o.items() if k[1] == "paired"}    # variant
    rows.append(dict(variant=vname(v), holds_a=any(x[0] and x[2] for x in ou.values()),
                     holds_b=any(x[1] and x[2] for x in ou.values()),
                     cos_a_any=any(x[0] for x in ou.values()), cos_b_any=any(x[1] for x in ou.values()),
                     hindered_any=any(x[2] for x in ou.values()),
                     hindered_poles=";".join(k[0] for k, x in ou.items() if x[2]) or "-",
                     holds_paired=any((x[0] or x[1]) and x[2] for x in op.values()),
                     A_150=round(fi["single"][0], 1), A_1000=round(fi[250][0], 1), A_1400=round(fi[350][0], 1),
                     R2_1000=round(fi[250][2], 2), R2_1400=round(fi[350][2], 2)))
t7a = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 80)
print(t7a.to_string())
print("\nclaim holds with the unpaired test (under a pole and cosine reading) in",
      int((t7a.holds_a | t7a.holds_b).sum()), "of", len(t7a), "variants; cosine by reading (a) in",
      int(t7a.cos_a_any.sum()), ", by reading (b) in", int(t7a.cos_b_any.sum()), "; hindered in",
      int(t7a.hindered_any.sum()), "; with the paired test (variant) the claim holds in", int(t7a.holds_paired.sum()))

**Result: reproduced under the main reading, an assumption.** With the edge trace, all six 5-us
cells, background removed and dark frames dropped, the cosine part holds by reading (b): with the
unpaired test no pole reading differs from the 150/150 benchmark after 360 to 1000 V/cm
conditioning (smallest p 0.059, mean of poles at 1000 V/cm). The hindrance at 1200 and 1400 V/cm
holds at the cathodic pole (unpaired p 0.022 and 0.0010) and for the mean of both poles (p 0.0013
and 0.0002), not at the anodic pole (p 0.086 at 1200 V/cm). The cosine amplitude falls from 12.6%
dF (benchmark, R^2 0.73) to 10.4% at 1000 V/cm (R^2 0.68) and 5.0% at 1400 V/cm (R^2 -0.55).
Reading (a), the s.e. band around A cos(theta - phi), fails in every variant, and under the main
reading at both poles at the control and at 1000 V/cm: the profiles carry a uniform negative offset (the baseline images fade, see the data check; its
source is not established) that the criterion's fit has no term for. With an offset term, a diagnostic outside the criterion, the fit
lies within the s.e. at both poles up to 1000 V/cm and not at 1200 or 1400 V/cm. Over 22 variants
the claim holds in 19; it fails in the 3 with the first round's peak trace, where the 1200 V/cm
hindrance is not significant at any pole (the first round's outcome, partially reproduced). With
the paired test, a variant, it holds in 21 of 22. The outcome rests on the edge trace: the 3 variants
that fail are the 3 with the first round's trace, so the 19 are not independent confirmations.
The window of the edge trace is a further choice; the section "Sensitivity to the trace window",
below, runs 4 to 12 pixels.

## fig7b-test-pulse-unchanged-to-760

> Polarization by the test pulse 6 us after the conditioning pulse does not differ from the
> undamaged control for conditioning pulses up to 760 V/cm, and is at about 80% of control after
> 1000 V/cm, a difference that is not significant.

A claim of absence on the paper's two-sided t-test, n = 5 (here 6), each pole. The claims file,
amended after the reader's answer (questions/001525.yaml), takes the test as the unpaired two-sample
Student's t-test of per-cell dF against the control group; the paired test and the one-sample test
of per-cell normalized values are variants. The 80% is the group mean at 1000 V/cm over the group
mean of the control, per pole reading.

In [ ]:
def unchanged_eval(poles, poles_=("cathodic", "anodic")):
    res = {}
    for pole in poles_:
        for test in ("unpaired", "paired", "one-sample"):
            res[(pole, test)] = {S.STRENGTH_V_CM[L]: test_vs_control(poles, L, pole, test)
                                 for L in (100, 150, 200, 250)}
    pct = {pole: pct_of_control(poles, 250, pole) for pole in POLES}
    return res, pct

r, pct = unchanged_eval(P0, POLES)    # the criterion tests each pole; the mean of poles is shown too
print("Main reading, p values against the control (360, 570, 760, 1000 V/cm); unpaired is the paper's test:")
for k, v in r.items():
    print(f"{k[0]:>9} {k[1]:>10}:", {s: round(float(p), 4) for s, p in v.items()},
          "-> none below 0.05" if min(v.values()) >= 0.05 else "-> differs")
print("\nPercent of control (group means), all strengths:")
for pole in POLES:
    print(f"{pole:>9}:", {S.STRENGTH_V_CM[L]: round(float(pct_of_control(P0, L, pole)), 1) for L in S.LEVELS})
def effect(poles, L, pole):
    """Group difference from control (% dF) with its unpaired s.e. and Cohen's d (pooled sd),
    and the paired d of the per-cell differences (variant)."""
    x, c = S.polarization(poles, L, pole), S.polarization(poles, "single", pole)
    sp = np.sqrt((x.var(ddof=1) + c.var(ddof=1)) / 2)
    dlt = x - c
    return x.mean() - c.mean(), sp * np.sqrt(1 / len(x) + 1 / len(c)), (x.mean() - c.mean()) / sp, dlt.mean() / dlt.std(ddof=1)
print("\nEffect sizes: difference from control (% dF, s.e.), unpaired Cohen's d, paired d (variant):")
for pole in ("cathodic", "anodic"):
    for L in (100, 150, 200, 250):
        m, se, d, dp = effect(P0, L, pole)
        print(f"  {pole} {S.STRENGTH_V_CM[L]}: {m:+.2f} (s.e. {se:.2f}), d = {d:+.2f}, paired d = {dp:+.2f}")

In [ ]:
rows = []
for v in VARIANTS:
    poles = poles_for(**vargs(v))
    r, pct = unchanged_eval(poles)
    row = dict(variant=vname(v))
    for test in ("unpaired", "paired", "one-sample"):
        row[f"absent_{test}"] = all(min(r[(pole, test)].values()) >= 0.05 for pole in ("cathodic", "anodic"))
        row[f"min_p_{test}"] = round(float(min(min(r[(pole, test)].values()) for pole in ("cathodic", "anodic"))), 4)
    row["differs_at"] = ";".join(f"{pole} {s_}" for pole in ("cathodic", "anodic")
                                 for s_, p_ in r[(pole, "unpaired")].items() if p_ < 0.05) or "-"
    row.update({f"pct1000_{p}": round(pct[p], 1) for p in POLES})
    rows.append(row)
t7b = pd.DataFrame(rows)
print(t7b.to_string())
print("\nabsence holds with the unpaired test (the paper's) in", int(t7b.absent_unpaired.sum()), "of", len(t7b),
      "variants; with the paired test (variant) in", int(t7b.absent_paired.sum()),
      "; with the one-sample test (variant) in", int(t7b["absent_one-sample"].sum()),
      "\n80 +- 8% met under some pole reading in",
      int(t7b[[f"pct1000_{p}" for p in POLES]].apply(lambda r: any(abs(x - 80) <= 8 for x in r), axis=1).sum()))

**Result: reproduced under the main reading, an assumption.** With the unpaired t-test no
conditioning strength from 360 to 1000 V/cm differs from control at either pole (smallest p 0.098,
anodic pole at 760 V/cm; cathodic p 0.21 to 0.99). The absence holds in 19 of 22 variants; in the
other 3 the anodic pole differs at 760 V/cm (p 0.027 to 0.046; control last without a background,
dark frames kept or dropped, and the five cells without c5). At 1000 V/cm the test response is
79.7% of control at the anodic pole, within 80 +- 8, and 57.5% at the cathodic pole and 71.9% for
their mean, outside it (unpaired p 0.21 and 0.24); in all 22 variants at least one pole reading is
within tolerance. Unpaired effect sizes up to 1000 V/cm run from d = -1.05 to +0.15. The paired
test, a variant, finds no difference at either pole under the main reading (smallest p 0.098) and
in 18 of 22 variants; the one-sample test in 21 of 22. For the mean of both poles, which the
criterion does not test for absence, the paired test gives p 0.018 at 760 V/cm and 0.016 at 1000
V/cm (one-sample p 0.027 and 0.015; unpaired p 0.14 and 0.059). The trace of c5 dips at the
cathodic pole in its 760 and 1000 V/cm files (data check), where its cathodic dF is -3.3% and
-1.9%; these values pull the cathodic pole towards a difference, and none is found (unpaired p
0.74 and 0.21). Without c5 the anodic pole differs at 760 V/cm (p 0.037). The window of the
trace is run in the section "Sensitivity to the trace window", below.

## fig7b-test-pulse-reduced-1200-1400

> Conditioning pulses of 1200 and 1400 V/cm reduce charging by the test pulse 6 us later to about
> 70% and 50% of control (p < 0.05 and p < 0.01).

Measures and test as above: the unpaired t-test against the control group (the paired test a
variant), pole readings (a) cathodic, (b) anodic, (c) mean. Reproduced when under one pole reading
both values are within 70 +- 7 and 50 +- 5 and the differences reach p < 0.05 and p < 0.01.

In [ ]:
def reduced_eval(poles):
    out = {}
    for pole in POLES:
        for test in TESTS:
            v12, v14 = pct_of_control(poles, 300, pole), pct_of_control(poles, 350, pole)
            p12, p14 = test_vs_control(poles, 300, pole, test), test_vs_control(poles, 350, pole, test)
            inside = abs(v12 - 70) <= 7 and abs(v14 - 50) <= 5
            sig = p12 < 0.05 and p14 < 0.01 and v12 < 100 and v14 < 100
            out[(pole, test)] = (v12, v14, p12, p14, inside, sig)
    return out

o = reduced_eval(P0)
print("Main reading: pole, test: % of control at 1200 and 1400 V/cm, p values, within tolerance, significant")
print("(unpaired is the paper's test; paired is a variant)")
for k, (v12, v14, p12, p14, ins, sig) in o.items():
    print(f"{k[0]:>9} {k[1]:>9}: {v12:6.1f} {v14:6.1f}  p = {p12:.4f}, {p14:.4f}  {ins} {sig}")
print("Effect sizes: difference from control (% dF, s.e.), unpaired Cohen's d, paired d (variant):")
for pole in ("cathodic", "anodic"):
    for L in (300, 350):
        m, se, d, dp = effect(P0, L, pole)
        print(f"  {pole} {S.STRENGTH_V_CM[L]}: {m:+.2f} (s.e. {se:.2f}), d = {d:+.2f}, paired d = {dp:+.2f}")

In [ ]:
rows = []
for v in VARIANTS:
    o = reduced_eval(poles_for(**vargs(v)))
    full = [k[0] for k, x in o.items() if k[1] == TEST and x[4] and x[5]]
    part = [k[0] for k, x in o.items() if k[1] == TEST and x[5]]
    row = dict(variant=vname(v), reproduced_under=";".join(full) or "-",
               significant_under=";".join(part) or "-",
               paired_variant=";".join(k[0] for k, x in o.items() if k[1] == "paired" and x[4] and x[5]) or "-")
    for pole in POLES:
        row[f"{pole}_1200"], row[f"{pole}_1400"] = round(o[(pole, TEST)][0], 1), round(o[(pole, TEST)][1], 1)
    row["p1200_anod"] = round(float(o[("anodic", TEST)][2]), 4)
    row["p1400_anod"] = round(float(o[("anodic", TEST)][3]), 4)
    rows.append(row)
t7r = pd.DataFrame(rows)
print(t7r.to_string())
print("\nunpaired test: reproduced under some pole reading in", int((t7r.reproduced_under != "-").sum()), "of",
      len(t7r), "variants; both significance levels reached (values outside tolerance) in",
      int(((t7r.significant_under != "-") & (t7r.reproduced_under == "-")).sum()),
      "| paired variant reproduced in", int((t7r.paired_variant != "-").sum()))
an = [reduced_eval(poles_for(**vargs(v)))[("anodic", TEST)] for v in VARIANTS]
print("anodic reading alone (the reader's question, questions/001525.yaml): values within tolerance in",
      sum(x[4] for x in an), "of", len(an), "variants; both significance levels in", sum(x[5] for x in an),
      "; the whole criterion in", sum(x[4] and x[5] for x in an))

**Result: partially reproduced under the main reading, an assumption.** Both reductions reach the
paper's significance levels under the cathodic reading (25.2% and -11.3% of control after 1200
and 1400 V/cm, unpaired p 0.022 and 0.0010) and for the mean of both poles (53.4% and 27.2%, p
0.0013 and 0.0002), but these values lie outside 70 +- 7 and 50 +- 5. The anodic pole has values
within tolerance (68.7% and 48.0%) but p 0.086 and 0.011, short of p < 0.05 and p < 0.01. No pole
reading meets the whole criterion in any of the 22 variants; both significance levels are reached
with a value outside tolerance, which the criterion calls partial, in 19; in the 3 with the first
round's peak trace no pole reading reaches both (the first round's outcome, not reproduced). The
paired variant meets the whole criterion in none. Effect sizes at 1400 V/cm: -10.5% dF at the
cathodic pole (unpaired d -2.66) and -9.1% dF at the anodic (d -1.80). The partial outcome comes
from the cathodic and mean readings only. Results 3.7 paragraph 6 omits one negative dF value,
"cathodic pole at 1400 V/cm", from Fig. 7C; if that is the test-pulse value, the paper's "~50%"
cannot be a cathodic or mean-of-poles value and only the anodic reading can carry it. Under the
anodic reading alone the values are within tolerance in 12 of 22 variants and both significance
levels are reached in 2, the whole criterion in none; under the main reading it is p 0.086 and
0.011, which would make the claim not reproduced. This is asked of a reader
(questions/001525.yaml); the outcome is judged on the criterion as written. The cathodic value
after 1400 V/cm (-11.3%) is negative, as the omitted value is; after 1200 V/cm (25.2%) it lies
well below the paper's 70%.

## fig7c-full-reseal-mild-residual-strong

> After mildly electroporating pulses (570 and 760 V/cm) the membrane reseals fully within 6 us,
> while after stronger pulses increased conductivity persists for at least 6 us; the cathodic pole
> is more conductive than the anodic during the pulse, but the test pulse finds no difference
> between the poles.

Four parts. Full reseal: test-pulse pole dF at 570 and 760 V/cm does not differ from control by
the test of fig7b-test-pulse-unchanged-to-760 (unpaired), each pole. Residual: test lm,eff above
the intact value (test dF below 100% of control, group means) at 1000, 1200 and 1400 V/cm, each
pole; lm,eff from Eq. (2) as in fig7c-conductivity-drops-10-fold (a ratio above 100% set to 100%;
a negative ratio, no charging, has no finite lm,eff and lies above the intact value). No pole
difference at the test pulse: paired two-sided t-test of cathodic against anodic normalized test
dF per strength (unpaired a variant; normalized by each pole's group control, and by the cell's
own control as a further variant). The pole difference during the pulse needs the
conditioning-pulse images of Fig. 4B, which the dataset does not hold, so the claim can be at most
partially reproduced (reseal and residual both hold).

In [ ]:
from scipy.optimize import brentq
f0 = S.f_factor(S.LAMBDA_M_INTACT)
def lm_eff(ratio):
    """lm,eff (S/m) at which Eq. (2) gives the measured fraction of the intact IMP (fig7c criterion:
    the test-pulse ratio is the mean pole dF over the control). A ratio above 100% is set to 100%
    (intact); a negative ratio (no charging) has no finite value."""
    if ratio <= 0:
        return np.inf
    r = min(ratio, 1.0)
    if r >= 1.0:
        return S.LAMBDA_M_INTACT
    return 10 ** brentq(lambda x: S.f_factor(10 ** x) / f0 - r, np.log10(S.LAMBDA_M_INTACT), 2)

def fig7c_eval(poles, verbose=False):
    reseal = {(pole, S.STRENGTH_V_CM[L]): test_vs_control(poles, L, pole, TEST)
              for pole in ("cathodic", "anodic") for L in (150, 200)}
    ratios = {(pole, S.STRENGTH_V_CM[L]): pct_of_control(poles, L, pole) / 100
              for pole in POLES for L in S.LEVELS[1:]}
    residual = {k: v < 1 for k, v in ratios.items() if k[1] >= 1000 and k[0] != "mean"}
    # no pole difference at the test pulse: cathodic against anodic normalized test dF per cell,
    # normalized by the group control of each pole (main), or by the cell's own control (variant)
    pdiff = {}
    for L in S.LEVELS[1:]:
        xc = 100 * S.polarization(poles, L, "cathodic") / S.polarization(poles, "single", "cathodic").mean()
        xa = 100 * S.polarization(poles, L, "anodic") / S.polarization(poles, "single", "anodic").mean()
        xc_own = 100 * S.polarization(poles, L, "cathodic") / S.polarization(poles, "single", "cathodic")
        xa_own = 100 * S.polarization(poles, L, "anodic") / S.polarization(poles, "single", "anodic")
        pdiff[S.STRENGTH_V_CM[L]] = (xc.mean() - xa.mean(), stats.ttest_rel(xc, xa).pvalue,
                                     stats.ttest_ind(xc, xa).pvalue, stats.ttest_rel(xc_own, xa_own).pvalue)
    ok_reseal = all(p >= 0.05 for p in reseal.values())
    ok_resid = all(residual.values())
    ok_pdiff = all(v[1] >= 0.05 for v in pdiff.values())
    # variant for the reader's question (questions/001525.yaml): Fig. 7C leaves out the negative
    # cathodic 1400 V/cm point, so the paper's comparison may cover 360 to 1200 V/cm only
    ok_pdiff_no1400 = all(v[1] >= 0.05 for s_, v in pdiff.items() if s_ < 1400)
    if verbose:
        print("Reseal, unpaired p against control:", {k: round(float(p), 4) for k, p in reseal.items()})
        print("\nTest-pulse ratio (group mean over control) and lm,eff (mS/m; intact 3e-4 mS/m):")
        for pole in POLES:
            print(f"{pole:>9}:", {s_: (round(float(ratios[(pole, s_)]) * 100, 1), float(f"{lm_eff(ratios[(pole, s_)]) * 1e3:.3g}"))
                                  for s_ in [S.STRENGTH_V_CM[L] for L in S.LEVELS[1:]]})
        print("\nResidual (test ratio below 100%) at 1000, 1200, 1400 V/cm, each pole:", {k: bool(v) for k, v in residual.items()})
        print("\nPole difference at the test pulse, cathodic minus anodic normalized dF (% of control), "
              "paired p (criterion), unpaired p (variant), paired p normalized per cell (variant):")
        for s_, (d_, pp, pu, po) in pdiff.items():
            print(f"  {s_:>5}: {d_:+7.1f}  paired p = {pp:.4f}  unpaired p = {pu:.4f}  per-cell paired p = {po:.4f}")
    if verbose:
        print("No pole difference at 360 to 1200 V/cm (1400 V/cm left out, variant):", ok_pdiff_no1400,
              "| smallest paired p", round(float(min(v[1] for s_, v in pdiff.items() if s_ < 1400)), 4))
    return ok_reseal, ok_resid, ok_pdiff, pdiff

ok_reseal, ok_resid, ok_pdiff, pdiff = fig7c_eval(P0, verbose=True)
print(f"\nMain reading: full reseal at 570 and 760 V/cm {ok_reseal}; residual at 1000 to 1400 V/cm {ok_resid}; "
      f"no pole difference at the test pulse {ok_pdiff}; pole difference during the pulse not testable (no conditioning-pulse images)")

In [ ]:
rows = []
for v in VARIANTS:
    r_, s_, d_, pdv = fig7c_eval(poles_for(**vargs(v)))
    rows.append(dict(variant=vname(v), reseal=r_, residual=s_, no_pole_diff_test=d_,
                     no_pole_diff_without_1400=all(x[1] >= 0.05 for k, x in pdv.items() if k < 1400),
                     pole_diff_at=";".join(str(k) for k, x in pdv.items() if x[1] < 0.05) or "-"))
t7c = pd.DataFrame(rows)
print(t7c.to_string())
print("\nreseal and residual both hold in", int((t7c.reseal & t7c.residual).sum()), "of", len(t7c),
      "variants; no pole difference at the test pulse in", int(t7c.no_pole_diff_test.sum()),
      "; with 1400 V/cm left out (variant) in", int(t7c.no_pole_diff_without_1400.sum()))

**Result: partially reproduced under the main reading, an assumption.** Full reseal: after 570 and
760 V/cm the test response does not differ from control at either pole (unpaired p 0.81 and 0.74
cathodic, 0.64 and 0.098 anodic), though the anodic pole is at 71.2% of control after 760 V/cm.
Residual: the test response is below 100% of control after 1000, 1200 and 1400 V/cm at both poles
(cathodic 57.5%, 25.2%, -11.3%; anodic 79.7%, 68.7%, 48.0%), a test lm,eff of 0.17 and 0.66 mS/m
and no finite value at the cathodic pole and 0.057, 0.10 and 0.24 mS/m at the anodic, against
3e-4 mS/m intact. Reseal and residual both hold in 19 of 22 variants; the 3 others are those in
which the anodic pole differs from control at 760 V/cm. The test pulse does find a pole
difference after 1400 V/cm (cathodic minus anodic -59.3 points of control, paired p = 0.0002),
where the cathodic pole no longer charges, so the part "no difference between the poles" fails
under the main reading; it holds only in the 6 variants that keep the dark frames with the edge
trace. Fig. 7C, to which the paper's sentence refers, leaves out the negative cathodic point at 1400 V/cm;
with 1400 V/cm left out, a variant, the paired test finds no pole difference at 360 to 1200 V/cm
(smallest p 0.12), in 21 of 22 variants. The criterion tests 1400 V/cm, and this is asked of a
reader (questions/001525.yaml); it does not change the outcome, which reseal and residual decide.
The pole difference during the pulse is not tested (no conditioning-pulse images). Under the
criterion, reseal and residual holding make the claim partially reproduced.

## Sensitivity to the trace window

The trace used is the intensity peak within 8 pixels inside the cell's outer edge, a width the
paper does not fix (it draws the trace by hand, Methods 2.3). Each Fig. 7 claim is run again with
windows of 4, 6, 10 and 12 pixels, under the main reading otherwise (dark frames dropped, annulus
background, control the mean of a cell's single-pulse files, field direction from that trace's own
profiles), with all six 5-us cells and without c5, the cell whose cathodic trace depends most on
the window (data check above).


In [ ]:
rows = []
for w in (4, 6, 8, 10, 12):
    key = MAIN if w == S.EDGE_WINDOW_PX else (f"dF_w{w}", True, True)
    if key not in PHI:
        PHI[key] = S.field_angle(res5, key)[0]
    for subs in (None, tuple(s for s in five if not s.endswith("_c5"))):
        poles = poles_for(key, list(subs) if subs else None)
        o7a, _ = fig7a_eval(key, list(subs) if subs else None)
        ou = {k: x for k, x in o7a.items() if k[1] == TEST}
        r_, pct_ = unchanged_eval(poles)
        o7r = reduced_eval(poles)
        rs, rd, nd, pdv = fig7c_eval(poles)
        rows.append(dict(window=w, cells="6" if subs is None else "without c5", phi=round(PHI[key], 1),
                         fig7a_holds=any(x[1] and x[2] for x in ou.values()),
                         hindered=";".join(k[0] for k, x in ou.items() if x[2]) or "-",
                         unchanged_absent=all(min(r_[(p_, "unpaired")].values()) >= 0.05 for p_ in ("cathodic", "anodic")),
                         unchanged_differs=";".join(f"{p_} {s_}" for p_ in ("cathodic", "anodic")
                                                    for s_, q in r_[(p_, "unpaired")].items() if q < 0.05) or "-",
                         unchanged_min_p=round(float(min(min(r_[(p_, "unpaired")].values()) for p_ in ("cathodic", "anodic"))), 3),
                         pct1000_c=round(pct_["cathodic"], 1), pct1000_a=round(pct_["anodic"], 1), pct1000_m=round(pct_["mean"], 1),
                         reduced_full=";".join(k[0] for k, x in o7r.items() if k[1] == TEST and x[4] and x[5]) or "-",
                         reduced_sig=";".join(k[0] for k, x in o7r.items() if k[1] == TEST and x[5]) or "-",
                         anod_1200=round(o7r[("anodic", TEST)][0], 1), anod_1400=round(o7r[("anodic", TEST)][1], 1),
                         anod_p1200=round(float(o7r[("anodic", TEST)][2]), 3), anod_p1400=round(float(o7r[("anodic", TEST)][3]), 4),
                         reseal_residual=rs and rd, no_pole_diff=nd,
                         pole_diff_p1400=round(float(pdv[1400][1]), 4)))
tw = pd.DataFrame(rows)
print(tw.to_string(index=False))
six = tw[tw.cells == "6"]
print(f"\nall six cells: fig7a holds with windows {six.window[six.fig7a_holds].tolist()}; fig7b-unchanged absence "
      f"with {six.window[six.unchanged_absent].tolist()}; fig7b-reduced both significance levels (partial or better) with "
      f"{six.window[six.reduced_sig != '-'].tolist()}, the whole criterion with {six.window[six.reduced_full != '-'].tolist()}; "
      f"fig7c reseal and residual with {six.window[six.reseal_residual].tolist()}")


**Result: the trace window.** With all six cells the Fig. 7A claim holds under every window from 4
to 12 pixels (hindered at the cathodic pole and the mean of poles from 6 to 10 px, at every pole
with 4 px, for the mean of poles only with 12 px). The fig7b-test-pulse-unchanged-to-760 absence
holds with 6 to 12 px and fails with 4 px (anodic pole at 760 V/cm, p 0.028); the 1000 V/cm anodic
value stays at 79.3 to 79.8% of control. fig7b-test-pulse-reduced-1200-1400 reaches both
significance levels with values outside tolerance (the criterion's partial) under every window and
meets the whole criterion under none; the anodic values stay within tolerance (67.3 to 69.2% and
43.6 to 48.3%) with p 0.035 to 0.092 at 1200 V/cm and 0.0021 to 0.011 at 1400 V/cm. The reseal and
residual parts of fig7c-full-reseal-mild-residual-strong hold with 6 to 12 px and fail with 4 px;
the pole difference at 1400 V/cm is found under every window (paired p 0.0002 to 0.0014). Without
c5, every window makes the fig7b-test-pulse-unchanged-to-760 absence fail at the anodic pole after
760 V/cm (p 0.010 to 0.046), as the main window does, and the Fig. 7A claim fails with 4 and 12 px
(hindered only at the anodic pole and the mean of poles; not hindered at the cathodic pole). The main reading's outcomes hold with windows
of 6 to 12 pixels; a 4-pixel window, which centres the 6-pixel-wide trace at most 4 px inside the
outer edge, would leave fig7a-test-pulse-cosine-up-to-1000 and fig7b-test-pulse-reduced-1200-1400
unchanged and fail the other two.


## fig5b-conductivity-calibration

> Under Eq. (2), effective membrane conductivity barely affects charging while it stays far below
> cytosol and medium conductivity; a 10-fold increase reduces IMP by about 2% and a 100-fold
> increase by 12%.

A computation from Eq. (2) with the Methods' parameters; no recording. The equation in the PDF's
text layer (page 3) has the form the claims file reads; the rendered page could not be viewed
here (no PDF renderer).

In [ ]:
lm0 = S.LAMBDA_M_INTACT
f0 = S.f_factor(lm0)
print(f"f for intact membrane (3e-7 S/m): {f0:.5f}")
for k in (10, 100, 1000, 1e4):
    f = S.f_factor(lm0 * k)
    print(f"{k:>7g}-fold (lm = {lm0*k*1e3:.4g} mS/m): reduction 1 - f/1.5 = {100*(1 - f/1.5):.2f}%, "
          f"relative to intact f = {100*(1 - f/f0):.2f}%")
lms = np.logspace(-7, 0, 400)
red = 100 * (1 - S.f_factor(lms) / 1.5)
fig, ax = plt.subplots(figsize=(4, 3))
ax.semilogx(lms * 1e3, 100 - red); ax.set_xlabel("lm, mS/m"); ax.set_ylabel("IMP, % of IMPC")
plt.show()
print(f"plotted IMP ranges {np.min(100 - red):.1f} to {np.max(100 - red):.1f}% of IMPC; "
      f"lm where IMP falls to 50%: {lms[np.argmin(abs(100 - red - 50))]*1e3:.3g} mS/m")
# Sensitivity of the 10-fold value to the parameters the Methods fix
for name, kw in [("R 6 um", dict(R=6e-6)), ("R 10 um", dict(R=10e-6)), ("d 7 nm", dict(d=7e-9))]:
    print(name, f"10-fold: {100*(1 - S.f_factor(lm0*10, **kw)/1.5):.2f}%, 100-fold: {100*(1 - S.f_factor(lm0*100, **kw)/1.5):.2f}%")

**Result: partially reproduced.** Eq. (2) with the Methods' parameters gives f = 1.498 for the
intact membrane, an IMP reduction of 11.87% for a 100-fold rise in membrane conductivity (within
12 +- 1.2) and 1.33% for a 10-fold rise, outside 2 +- 0.5 (1.20% relative to the intact f). Within the
ranges tried for R (6 to 10 um) the 10-fold reduction stays between 1.00% and 1.65%.

## Summary

| Claim | Outcome |
|---|---|
| fig7a-test-pulse-cosine-up-to-1000 | reproduced, resting on assumptions (fails with the first round's peak trace) |
| fig7b-test-pulse-unchanged-to-760 | reproduced, resting on assumptions (anodic 760 V/cm differs in 3 of 22 variants) |
| fig7b-test-pulse-reduced-1200-1400 | partially reproduced, resting on assumptions (significant, values outside tolerance) |
| fig7c-full-reseal-mild-residual-strong | partially reproduced, resting on assumptions (reseal and residual hold; a pole difference at 1400 V/cm; the during-pulse part not testable) |
| fig5b-conductivity-calibration | partially reproduced |
| fig7b-conditioning-pulse-below-linear | blocked: data missing |
| fig7c-conductivity-drops-10-fold | blocked: data missing (depends on the claim above) |
| fig1, fig2 (6), fig3 (2), fig4 (6), fig5a (2), fig5c, fig6 (2): 20 claims | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The perimeter trace is found automatically, as the
intensity peak inside the cell's outer edge, not drawn by hand; the first round's trace, the plain
intensity peak, fell on bright dye inside two cells and is kept as a variant, under which the Fig.
7A hindrance at 1200 V/cm is not significant. The window of the trace used (8 pixels) is a
choice; windows of 6 to 12 pixels give the same outcomes, and 4 pixels fails
fig7b-test-pulse-unchanged-to-760 and the reseal and residual test of fig7c-full-reseal-mild-residual-strong. The cell is the largest object in the field; the
field direction is taken from the data; the conditioning strengths are mapped to the files' volts
by rank; the benchmark is the dataset's single 50 V pulse, recorded first in every cell, where the
paper describes a 150/150 pair; six cells are used where the paper has five; a background and the
dark frames are handled under two readings each, with the background-subtracted, dark-dropped
reading adopted. The Fig. 7 tests are the unpaired two-sample t-test against the control group, as
the claims file has it since the reader's answer (questions/001525.yaml); the paired test is shown
as a variant.

**What was found about the dataset.** It holds only the two-pulse experiment of Fig. 7 (10 cells,
of which 6 at the paper's 5-us interval; the other 4 at a 1-us interval the paper does not report).
The files give pulses in generator volts, not field strengths, and an electrode gap of 1.00 mm where
the paper gives 1.6 mm; the shared description lists field strengths (0.15 to 1.57 kV/cm) and
intervals (1, 5, 25, 125 us) that the files do not hold. The 150/150 benchmark of the paper is a
single pulse in the files. Most files have a dark first image (no laser flash), and the series
description says exposures began 1 us after the start of recording, while the images show the
pulse from image 40. The images fade by about 0.1% per frame; the
profiles carry a uniform negative offset of the same sign, whose source is not established. No recording is deposited twice.